# 微信聊天角色扮演 - Qwen3.5-9B 微调

使用LoRA微调Qwen3.5-9B，让模型学习模仿聊天记录中对方的说话风格。

## 环境要求
- Google Colab (GPU)
- 免费版T4 GPU即可运行

In [ ]:
# 安装依赖
! pip install -q -U transformers peft trl datasets baccelerate bitsandbytes

In [ ]:
import json
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# 检查GPU
print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. 上传训练数据

将预处理好的 `train_data.json` 和 `config.json` 上传到Colab

In [ ]:
from google.colab import files

# 上传训练数据和配置文件
uploaded = files.upload()
data_file = [f for f in uploaded.keys() if 'train_data' in f][0]
config_file = [f for f in uploaded.keys() if 'config' in f][0]
print(f"已上传: {data_file}, {config_file}")

In [ ]:
# 加载训练数据和配置
with open(data_file, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(config_file, 'r', encoding='utf-8') as f:
    config = json.load(f)

system_prompt = config.get("system_prompt", "")

# 注入 system prompt 到每条对话开头
if system_prompt:
    for sample in train_data:
        conversations = sample["conversations"]
        if conversations[0]["role"] != "system":
            conversations.insert(0, {"role": "system", "content": system_prompt})
    print(f"已注入 system prompt: {system_prompt}")
else:
    print("未设置 system prompt，跳过注入")

print(f"训练样本数量: {len(train_data)}")
print(f"\n样本示例:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

## 2. 加载模型和Tokenizer

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# 设置pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"词表大小: {len(tokenizer)}")
print(f"特殊token: pad={tokenizer.pad_token}, eos={tokenizer.eos_token}")

In [ ]:
# 4-bit量化配置（节省显存）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # Qwen原生BF16
    bnb_4bit_use_double_quant=True
)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 准备模型进行k-bit训练
model = prepare_model_for_kbit_training(model)

print(f"模型加载完成")
print(f"模型参数量: {model.num_parameters() / 1e6:.1f}M")

## 3. 配置LoRA

In [ ]:
# LoRA配置
# 数据量极小（29条对话，~1400有效token），降低LoRA容量防止过拟合
lora_config = LoraConfig(
    r=8,                     # LoRA秩：从16降至8，小数据无需高秩
    lora_alpha=16,           # 缩放系数：保持 alpha/r = 2
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # 仅注意力层，去掉FFN减少参数
    ],
    lora_dropout=0.1,        # 从0.05提高到0.1，增强正则化
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用LoRA
model = get_peft_model(model, lora_config)

# 打印可训练参数
model.print_trainable_parameters()

## 4. 准备数据集

In [ ]:
def format_conversation(example):
    """
    将对话格式化为Qwen3的chat模板（禁用thinking，避免<think>标签污染聊天风格）
    """
    messages = example["conversations"]

    # 使用tokenizer的chat模板，禁用thinking
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    return {"text": text}

# 创建Dataset
dataset = Dataset.from_list(train_data)
dataset = dataset.map(format_conversation)

print(f"数据集大小: {len(dataset)}")
print(f"\n格式化后的样本:")
print(dataset[0]["text"][:500])

In [ ]:
# 展示格式化后的训练样本，了解实际输入模型的文本格式
num_preview = min(3, len(dataset))
for i in range(num_preview):
    print(f"{'='*60}")
    print(f"样本 {i+1}/{len(dataset)}（共 {len(dataset[i]['text'])} 字符）")
    print(f"{'='*60}")
    print(dataset[i]["text"])
    print()

## 5. 训练配置

In [ ]:
# 训练参数
# 针对极小数据集（29条对话）优化：多epoch、小batch、适度学习率
training_args = SFTConfig(
    output_dir="./output",
    num_train_epochs=5,                  # 从2提高到5：数据少需要多轮
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,       # 从16降至4：有效batch=4，保证每epoch有足够更新步
    learning_rate=2e-4,                  # 从1e-4提至2e-4：配合小batch和LoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,                   # 从0.1降至0.05：总步数少，快速进入训练
    bf16=True,                           # Qwen原生BF16，与量化compute_dtype保持一致
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,                       # 每10步保存，便于挑选最优checkpoint
    save_total_limit=3,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    dataset_text_field="text",
    packing=False,
    # 仅对 assistant 回复部分计算 loss（替代已弃用的 DataCollatorForCompletionOnlyLM）
    completion_only_loss=True,
    response_template="<|im_start|>assistant\n",
    instruction_template="<|im_start|>",
)

# 通过tokenizer控制序列长度
tokenizer.model_max_length = 256         # 从512降至256：最长对话177字符，256足够

print("训练配置:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Total steps (estimated): {int(29 / 4 * 5)}")
print(f"  Warmup steps (estimated): {max(1, int(29 / 4 * 5 * 0.05))}")

In [ ]:
# 创建Trainer
# completion_only_loss=True 已在 SFTConfig 中配置，自动实现仅对 assistant 回复计算 loss
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Trainer已创建，准备开始训练")
print("已启用 assistant-only masking，仅对 assistant 回复计算 loss")

## 6. 开始训练

In [82]:
# 开始训练
print("开始训练...")
trainer.train()
print("训练完成！")

开始训练...


Step,Training Loss
1,0.022376
2,0.021882


训练完成！


## 7. 保存模型

In [ ]:
# 保存LoRA适配器
LORA_OUTPUT_DIR = "./lora_adapter"
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# 将 config.json 一并保存到 LoRA 目录，确保推理时使用相同的 system prompt
import shutil
shutil.copy(config_file, f"{LORA_OUTPUT_DIR}/config_chat.json")

print(f"LoRA适配器已保存至: {LORA_OUTPUT_DIR}")
print(f"config_chat.json 已一并保存（包含 system prompt）")

In [ ]:
# （可选）合并为完整模型
# 注意：9B FP16 模型约18GB，Colab免费版内存可能不足
# 如果内存不够，建议跳过此步骤，直接使用 LoRA 适配器推理
import psutil

MERGE_OUTPUT_DIR = "./merged_model"

ram_gb = psutil.virtual_memory().total / 1e9
print(f"当前可用内存: {ram_gb:.1f} GB")

if ram_gb < 20:
    print("⚠️ 内存不足20GB，合并9B模型可能会OOM。")
    print("建议跳过此步骤，直接使用 LoRA 适配器推理。")
    print("如需合并，请使用 Colab Pro（高内存运行时）。")
else:
    # 重新加载基座模型（不量化）
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="cpu",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )

    # 加载LoRA权重并合并
    from peft import PeftModel
    merged_model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_DIR)
    merged_model = merged_model.merge_and_unload()

    # 保存合并后的模型
    merged_model.save_pretrained(MERGE_OUTPUT_DIR)
    tokenizer.save_pretrained(MERGE_OUTPUT_DIR)

    print(f"合并后的模型已保存至: {MERGE_OUTPUT_DIR}")

## 8. 测试模型

In [ ]:
def chat(model, tokenizer, user_input, history=[]):
    """
    单轮对话测试
    """
    messages = history + [{"role": "user", "content": user_input}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()

In [ ]:
# 测试对话
test_inputs = [
    "在干嘛",
    "今天累不累",
    "晚上吃什么",
    "一起看电影吧"
]

# 使用与训练一致的 system prompt
test_history = []
if system_prompt:
    test_history = [{"role": "system", "content": system_prompt}]

print("=== 模型测试 ===")
if system_prompt:
    print(f"[system prompt: {system_prompt}]\n")
for user_input in test_inputs:
    response = chat(merged_model, tokenizer, user_input, history=test_history)
    print(f"用户: {user_input}")
    print(f"模型: {response}")
    print()

## 9. 下载模型

将训练好的模型下载到本地

In [87]:
# 打包LoRA适配器
!zip -r lora_adapter.zip ./lora_adapter

# 下载
from google.colab import files
files.download('lora_adapter.zip')

print("LoRA适配器已下载，解压后放到本地项目目录使用")

updating: lora_adapter/ (stored 0%)
updating: lora_adapter/adapter_config.json (deflated 59%)
updating: lora_adapter/README.md (deflated 65%)
updating: lora_adapter/chat_template.jinja (deflated 76%)
updating: lora_adapter/tokenizer_config.json (deflated 59%)
updating: lora_adapter/adapter_model.safetensors (deflated 21%)
updating: lora_adapter/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

LoRA适配器已下载，解压后放到本地项目目录使用


In [88]:
# （可选）打包合并后的完整模型
# 注意：完整模型文件较大，下载可能需要较长时间
!zip -r merged_model.zip ./merged_model

from google.colab import files
files.download('merged_model.zip')

updating: merged_model/ (stored 0%)
updating: merged_model/chat_template.jinja (deflated 76%)
updating: merged_model/config.json (deflated 71%)
updating: merged_model/model.safetensors (deflated 14%)
updating: merged_model/tokenizer_config.json (deflated 59%)
updating: merged_model/generation_config.json (deflated 38%)
updating: merged_model/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 本地使用方法

```bash
# 方法1: 使用LoRA适配器
python inference/chat.py --model Qwen/Qwen3.5-9B --lora ./lora_adapter

# 方法2: 使用合并后的模型
python inference/chat.py --model ./merged_model
```